# IOAI — 2025 Stage 1 Label Noise (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/train/dataset_labels.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-1-label-noise/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 라벨 노이즈 — 코티칭 (Label Noise / Co-teaching, 모범답안)

폴란드 AI 올림피아드 II · 2025 · 1단계. 학습 라벨에 **잡음(오라벨)** 이 섞이고 **클래스 불균형**인 이진
이미지 분류. 아키텍처(`SmallMobileNet`)는 **고정**, **두 개**의 네트워크를 동시에 학습한다. 여러분은
`your_select_indices(targets, losses)` 만 구현 — 배치에서 **어떤 샘플로 각 모델을 학습할지** 인덱스를 고른다.

**힌트(수수께끼)**: 왜 모델이 둘일까? → **작은 손실 = 깨끗한 라벨**일 가능성이 높고, 각 모델이 상대 모델에게
깨끗해 보이는 샘플을 골라주면(코티칭) 잡음에 덜 오염된다. 불균형은 클래스별 선택비율로 다룬다.

**채점**: 두 모델의 **balanced accuracy** 평균 → `performance`(≤0.5→0, 0.5~0.8 선형, ≥0.8→100).

**모범답안**: 클래스별로 (불균형 대응) 상대 모델의 손실 기준 **작은 손실 샘플**만 골라 학습(코티칭).
클래스0 50%·클래스1 93% 채택. → BAC 평균 ≈ **0.79** → **97점** (베이스라인 0점).

**제출**: `submission.csv` — `file_name,pred1,pred2` (val 1000장에 대한 두 모델 예측).


In [ ]:
# 환경 + 데이터 (Colab: 자동 다운로드 / DGX: data/ 이미 존재). train=data/train, val=data/val
FINAL_EVALUATION_MODE = True   # 고정 train() 의 val-eval/플롯(정답 필요)을 끔 — val 정답은 채점 서버에만
import os, urllib.request, zipfile
if not os.path.exists("data/train/dataset_labels.csv"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-1-label-noise/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
import os
from tqdm import tqdm
from typing import Optional, Tuple, List

import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader

import torchvision.transforms as transforms
from torchvision.datasets.folder import VisionDataset

from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import balanced_accuracy_score
SEED = 123; IMAGES_DIR = "data"; TASK_DATASET_LABELS_FILE = "dataset_labels.csv"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 1e-2; NUM_EPOCHS = 6; NUM_CLASSES = 2; BATCH_SIZE = 128; WEIGHT_DECAY = 1e-3
print("device", DEVICE)


In [ ]:
######## 고정 코드(아키텍처·학습·평가) — 원문제와 동일 ########
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
def seed_everything(seed: int) -> None:
    """
    Ustawia ziarno (seed) dla odtwarzalności wyników w Pythonie, NumPy oraz PyTorch.

    Funkcja ustawia ziarno dla generatorów liczb losowych Pythonie, NumPy oraz PyTorch,
    a także konfiguruje PyTorch do pracy w trybie deterministycznym.

    Parametry:
        seed (int): Wartość ziarna do ustawienia.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# Klasa zbioru danych
class TaskDataset(VisionDataset):
    def __init__(
        self,
        root: str,
        transform: Optional[callable] = None,
    ):
        super().__init__(
            root,
            transform=transform,
        )
        self.root = root

        if not self._check_integrity():
            raise RuntimeError(
                f"Nie znaleziono zbioru danych. Sprawdź czy ścieżka {self.root} istnieje. Powinna ona zwierać folder '{IMAGES_DIR}' i plik '{TASK_DATASET_LABELS_FILE}' file"
            )
        self.labels_df = self._read_labels_from_file()
        self.labels_header = 'label'

    def _read_labels_from_file(self) -> pd.DataFrame:
        df = pd.read_csv(os.path.join(self.root, TASK_DATASET_LABELS_FILE))
        return df

    def _check_integrity(self) -> bool:
        return os.path.exists(os.path.join(self.root, IMAGES_DIR)) and os.path.exists(
            os.path.join(self.root, TASK_DATASET_LABELS_FILE)
        )

    def __len__(self) -> int:
        return len(self.labels_df)

    def __getitem__(self, idx: int) -> Tuple[Image.Image, np.ndarray]:
        img = self._load_image(idx)
        label = self._load_label(idx)
        if self.transform is not None:
            img = self.transform(img)
        return img, label

    def _load_image(self, idx: int) -> Image.Image:
        img_path = os.path.join(
            self.root, IMAGES_DIR, self.labels_df.iloc[idx]['file_name']
        )
        img = Image.open(img_path)
        return img

    def _load_label(self, idx: int):
        label = self.labels_df.iloc[idx][self.labels_header]
        return np.array([int(label)])

######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
class SmallMobileNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(SmallMobileNet, self).__init__()

        # Główne bloki konwolucyjne
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            nn.Conv2d(
                32, 32, kernel_size=3, stride=1, padding=1, groups=32, bias=False
            ),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            nn.Conv2d(32, 64, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU6(inplace=True),
            nn.Conv2d(
                64, 64, kernel_size=3, stride=2, padding=1, groups=64, bias=False
            ),
            nn.BatchNorm2d(64),
            nn.ReLU6(inplace=True),
            nn.Conv2d(64, 128, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU6(inplace=True),
            nn.Conv2d(
                128, 128, kernel_size=3, stride=2, padding=1, groups=128, bias=False
            ),
            nn.BatchNorm2d(128),
            nn.ReLU6(inplace=True),
            nn.Conv2d(128, 256, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU6(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU6(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
def predict_and_evaluate(model, val_loader, device, verbose=False):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    balanced_accuracy = balanced_accuracy_score(all_targets, all_preds)

    if verbose:
        print(f"Balanced Accuracy: {balanced_accuracy}")

    return balanced_accuracy

######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################


# Funkcja do trenowania modelu
def train(
    model1,
    model2,
    optimizer1,
    optimizer2,
    criterion,
    train_loader,
    val_loader,
    num_epochs,
    device,
    select_indices_fn,
):

    verbose = False if FINAL_EVALUATION_MODE else True

    # Historia metryk dla każdego modelu
    metrics = {
        k: [[], []]
        for k in [
            "train_loss",
            "val_loss",
            "train_bac",
            "val_bac",
        ]
    }
    epochs_range = np.arange(num_epochs) + 1

    # Główna pętla treningowa
    for epoch in epochs_range:
        print(f"Epoch {epoch}")

        # Historia statystyk dla każdego modelu
        stats = {
            k: [0, 0] for k in ["train_loss", "train_total", "val_loss", "val_total"]
        }
        preds_targets = {
            k: [[], []]
            for k in ["train_preds", "train_targets", "val_preds", "val_targets"]
        }

        model1.train(), model2.train()
        for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}"):
            inputs, targets = inputs.to(device), targets.squeeze().long().to(device)

            outputs = [m(inputs) for m in (model1, model2)]
            losses = [criterion(out, targets) for out in outputs]

            # --- GŁÓWNY PUNKT ZADANIA ---
            selected_indices = select_indices_fn(targets, losses)
            # ---------------------------

            # Propagacja wsteczna dla każdego modelu
            for i, (model, optim) in enumerate(
                [(model1, optimizer1), (model2, optimizer2)]
            ):
                optim.zero_grad()
                sel_idx = selected_indices[i]
                loss = criterion(model(inputs[sel_idx]), targets[sel_idx]).mean()

                loss.backward()
                optim.step()

                # Historia statystyk
                stats["train_loss"][i] += loss.item() * len(sel_idx)
                stats["train_total"][i] += len(sel_idx)
                preds = outputs[i].max(1)[1]
                preds_targets["train_preds"][i].extend(preds[sel_idx].cpu().numpy())
                preds_targets["train_targets"][i].extend(targets[sel_idx].cpu().numpy())

        # Ewaluacja na zbiorze walidacyjnym
        if verbose:
            model1.eval(), model2.eval()
            with torch.no_grad():
                for inputs, targets in tqdm(
                    val_loader, desc=f"Validation {epoch}/{num_epochs}"
                ):
                    inputs, targets = inputs.to(device), targets.squeeze().long().to(
                        device
                    )

                    for i, model in enumerate([model1, model2]):
                        outputs = model(inputs)
                        loss = criterion(outputs, targets).mean()
                        preds = outputs.max(1)[1]

                        stats["val_loss"][i] += loss.item() * inputs.size(0)
                        stats["val_total"][i] += inputs.size(0)
                        preds = outputs.max(1)[1]
                        preds_targets["val_preds"][i].extend(preds.cpu().numpy())
                        preds_targets["val_targets"][i].extend(targets.cpu().numpy())

        # Obliczanie metryk
        if verbose:
            models = [model1, model2]
            for i in range(2):
                for phase in ["train", "val"]:
                    preds = preds_targets[f"{phase}_preds"][i]
                    targets = preds_targets[f"{phase}_targets"][i]

                    metrics[f"{phase}_loss"][i].append(
                        stats[f"{phase}_loss"][i] / stats[f"{phase}_total"][i]
                    )
                    metrics[f"{phase}_bac"][i].append(
                        balanced_accuracy_score(targets, preds)
                    )

                print(
                    f"Model{i+1} - Train Loss: {metrics['train_loss'][i][-1]:.4f}, "
                    f"Train balanced accuracy: {metrics['train_bac'][i][-1]:.4f} --- "
                    f"Validation Loss: {metrics['val_loss'][i][-1]:.4f}, "
                    f"Validation balanced accuracy: {metrics['val_bac'][i][-1]:.4f}, "
                )

    # Generowanie wykresów
    if verbose:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        colors = ["#fa2729", "#ac1a1c", "#1a6aff", "#144aad"]
        linestyles = ["-", "--"]

        for i, model_name in enumerate(["Model1", "Model2"]):
            for j, phase in enumerate(["train", "val"]):

                color = colors[i * 2 + j]
                ax1.plot(
                    epochs_range,
                    metrics[f"{phase}_loss"][i],
                    color=color,
                    marker="o",
                    linestyle=linestyles[j],
                    label=f"{model_name} {phase.title()} Loss",
                )
                ax2.plot(
                    epochs_range,
                    metrics[f"{phase}_bac"][i],
                    color=color,
                    marker="o",
                    linestyle=linestyles[j],
                    label=f"{model_name} {phase.title()} balanced accuracy",
                )

        for ax, title in zip([ax1, ax2], ["Loss", "Balanced accuracy"]):
            ax.set_title(f"Training and Validation {title}")
            ax.set_xticks(epochs_range)
            ax.set_xlabel("Epochs")
            ax.set_ylabel(title)
            ax.legend()

        plt.tight_layout()
        plt.show()

In [ ]:
# 데이터 로더 (train=잡음·불균형 / val=예측대상, 정답 미공개)
base_transform = transforms.Compose([transforms.ToTensor()])
train_loader = DataLoader(TaskDataset(root="data/train", transform=base_transform), batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(TaskDataset(root="data/val",   transform=base_transform), batch_size=BATCH_SIZE, shuffle=False)
print("train batches", len(train_loader), "val batches", len(val_loader))


In [ ]:
def your_select_indices(targets, losses):
    """모범답안(코티칭): 클래스별로 상대 모델의 손실이 가장 작은(=깨끗할 가능성 높은) 샘플만 채택.
    클래스별 채택비율로 불균형까지 대응. 교차선택(losses[1-i])으로 두 모델이 서로를 가르친다."""
    take_n_by_class = [0.5, 0.93]              # 클래스0 50%, 클래스1 93% (실험으로 선택)
    selected = [[], []]
    for c in torch.unique(targets):
        idx = (targets == c).nonzero(as_tuple=True)[0]
        k = int(len(idx) * take_n_by_class[c])
        for i in range(2):
            t = losses[1 - i].clone().to(DEVICE)     # 상대 모델 손실
            t[targets.to(DEVICE) != c] = float("Inf")
            _, best = torch.topk(-t, k=k)            # 소손실 top-k
            selected[i].extend(best.cpu().tolist())
    return [torch.tensor(selected[0]).to(DEVICE), torch.tensor(selected[1]).to(DEVICE)]


In [ ]:
# 두 모델 학습 (고정 절차) + val 예측 -> submission.csv
seed_everything(SEED)
criterion = nn.CrossEntropyLoss(reduction="none")
model1 = SmallMobileNet(NUM_CLASSES).to(DEVICE); model2 = SmallMobileNet(NUM_CLASSES).to(DEVICE)
optimizer1 = AdamW(model1.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
optimizer2 = AdamW(model2.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
seed_everything(SEED)
train(model1, model2, optimizer1, optimizer2, criterion, train_loader, val_loader, NUM_EPOCHS, DEVICE,
      select_indices_fn=your_select_indices)

import csv
val_ds = val_loader.dataset; file_names = list(val_ds.labels_df["file_name"])
model1.eval(); model2.eval(); preds1 = []; preds2 = []
with torch.no_grad():
    for inputs, _ in val_loader:
        inputs = inputs.to(DEVICE)
        preds1.extend(model1(inputs).argmax(1).cpu().tolist())
        preds2.extend(model2(inputs).argmax(1).cpu().tolist())
with open("submission.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["file_name", "pred1", "pred2"])
    for fn, a, b in zip(file_names, preds1, preds2): w.writerow([fn, a, b])
print("submission.csv 저장:", len(file_names), "행")


### 정리
- 코티칭(클래스별 교차손실 소손실 선택) → 두 모델 BAC 평균 ≈ 0.79 → 97점 (베이스라인 0점).
- **핵심**: 작은 손실 = 깨끗한 라벨. 두 모델이 서로에게 깨끗한 샘플을 골라줘 잡음 기억을 늦춘다.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)